# QuantSupport Python bindings — a guided tour

This notebook walks through every component of the `quantsupport` Python API:

1. **Dates, periods and calendars** — the time toolkit
2. **Enums** — currencies, market indices, conventions
3. **Market data** — quote stores, curve configurations, FX
4. **The pricing context** — bootstrapping and lifecycle
5. **Exploring bootstrapped curves** — nodes, discount factors, forward rates, pillars
6. **Pricing trades** — swaps and cross-currency swaps with AD sensitivities
7. **XVA** — per-client CSA terms, netting sets, Monte Carlo exposures

Build and install the bindings first (from the repo root):

```bash
pip install maturin
maturin develop --release -m bindings/python/Cargo.toml
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import quantsupport as qs

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

# Market data JSON files shipped with the Rust CVA example.
DATA = next(
    p / "examples" / "cva" / "data"
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "examples" / "cva" / "data").exists()
)
DATA

PosixPath('/Users/josemelo/Desktop/dev/quantsupport/examples/cva/data')

## 1. Dates, periods and calendars

`qs.Date` is an immutable calendar date. It supports arithmetic with integers
(days), `qs.Period` objects, or period strings like `"6M"` / `"1Y6M"`.
Subtracting two dates gives the number of days between them.

In [2]:
d = qs.Date(2025, 11, 11)          # or qs.Date.parse("2025-11-11")
print(d, "is a", d.weekday())
print("+30 days:      ", d + 30)
print("+6 months:     ", d + "6M")
print("+1 year 6 mo:  ", d + qs.Period.parse("1Y6M"))
print("end of month:  ", d.end_of_month())
print("days to +6M:   ", (d + "6M") - d)

2025-11-11 is a Tuesday
+30 days:       2025-12-11
+6 months:      2026-05-11
+1 year 6 mo:   2027-05-11
end of month:   2025-11-30
days to +6M:    181


In [3]:
p = qs.Period(18, qs.TimeUnit.Months)
print(p, "->", p.length, p.units)
print("from frequency:", qs.Period.from_frequency(qs.Frequency.Quarterly))

18M -> 18 Months
from frequency: 3M


`qs.Calendar` answers business-day questions: adjusting dates with a
`BusinessDayConvention`, advancing by periods while skipping holidays, and
listing holidays or business days. `qs.DayCounter` measures year fractions
between dates under a market convention.

In [4]:
cal = qs.Calendar("UnitedStates")   # also: TARGET, Brazil, Chile, WeekendsOnly, NullCalendar
thanksgiving = qs.Date(2025, 11, 27)
print("business day?          ", cal.is_business_day(thanksgiving))
print("adjusted (Following):  ", cal.adjust(thanksgiving, qs.BusinessDayConvention.Following))
print("advance 5 business d.: ", cal.advance(d, "5D"))
print("business days in Nov:  ", cal.business_days_between(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))
print("holidays in Nov 2025:  ", cal.holiday_list(qs.Date(2025, 11, 1), qs.Date(2025, 11, 30)))

dc = qs.DayCounter.Actual360
print("Act/360 year fraction: ", dc.year_fraction(d, d + "6M"))

business day?           True
adjusted (Following):   2025-11-27
advance 5 business d.:  2026-07-03
business days in Nov:   1
holidays in Nov 2025:   [Date('2025-11-03'), Date('2025-11-04'), Date('2025-11-05'), Date('2025-11-06'), Date('2025-11-07'), Date('2025-11-10'), Date('2025-11-11'), Date('2025-11-12'), Date('2025-11-13'), Date('2025-11-14'), Date('2025-11-17'), Date('2025-11-18'), Date('2025-11-19'), Date('2025-11-20'), Date('2025-11-21'), Date('2025-11-24'), Date('2025-11-25'), Date('2025-11-26'), Date('2025-11-28')]
Act/360 year fraction:  0.5027777777777778


## 2. Enums

All conventions are proper Python enum classes (hashable, comparable, usable as
dict keys). Every API that takes an enum also accepts its string name, and each
enum has a case-insensitive `parse` static method.

`qs.Currency` carries full ISO 4217 metadata:

In [5]:
import pandas as pd

pd.DataFrame(
    {
        "code": c.code,
        "name": c.name,
        "symbol": c.symbol,
        "precision": c.precision,
        "numeric_code": c.numeric_code,
    }
    for c in [qs.Currency.USD, qs.Currency.EUR, qs.Currency.CLP, qs.Currency.JPY, qs.Currency.BRL]
)

,code,name,symbol,precision,numeric_code
0,USD,US Dollar,$,2,840
1,EUR,Euro,€,2,978
2,CLP,Chilean Peso,$,0,152
3,JPY,Japanese Yen,¥,0,392
4,BRL,Brazilian Real,R$,2,986


`qs.MarketIndex` identifies every curve, surface and simulation: overnight
rates (`SOFR`, `ESTR`, `ICP`, ...), term rates (`EURIBOR6m`, `TermSOFR3m`, ...),
plus parameterised indices built with static factories.

In [6]:
print(qs.MarketIndex.SOFR, "|", qs.MarketIndex.EURIBOR6m, "|", qs.MarketIndex.ICP)
print("equity:    ", qs.MarketIndex.equity("AAPL"))
print("fx pair:   ", qs.MarketIndex.fx_pair("CLP", "USD"))
print("collateral:", qs.MarketIndex.collateral("CLP", "USD"))
print("parse:     ", qs.MarketIndex.parse("sofr"))

# Other conventions follow the same pattern:
print(qs.Side.LongReceive.sign(), qs.Side.parse("pay"))
print(qs.Compounding.Compounded, qs.Frequency.Semiannual, qs.Request.Sensitivities)

SOFR | EURIBOR6m | ICP
equity:     AAPL
fx pair:    FX:CLPUSD
collateral: Collateral(CLP/USD)
parse:      SOFR
1.0 PayShort
Compounded Semiannual Sensitivities


## 3. Market data

A `qs.QuoteStore` holds all market quotes as of a single reference date. It
deserializes straight from JSON and can be inspected as a DataFrame.

In [7]:
quotes = qs.QuoteStore.from_json(str(DATA / "quotes.json"))
print("reference date:", quotes.reference_date, "| quotes:", len(quotes.identifiers()))
quotes.to_dataframe().head(8)

reference date: 2025-11-11 | quotes: 46


,identifier,mid,bid,ask
0,FixFloatCrossCurrencySwap_CLP_SOFR_USD_10Y,0.0440,None,None
1,FixFloatCrossCurrencySwap_CLP_SOFR_USD_1Y,0.0510,None,None
2,FixFloatCrossCurrencySwap_CLP_SOFR_USD_20Y,0.0430,None,None
3,FixFloatCrossCurrencySwap_CLP_SOFR_USD_2Y,0.0485,None,None
4,FixFloatCrossCurrencySwap_CLP_SOFR_USD_3Y,0.0465,None,None
5,FixFloatCrossCurrencySwap_CLP_SOFR_USD_5Y,0.0455,None,None
6,FixFloatCrossCurrencySwap_CLP_SOFR_USD_7Y,0.0450,None,None
7,FixedRateDeposit_CLP_ICP_1D,0.0500,None,None


`qs.CurveConfiguration` describes how each curve is bootstrapped: which market
index it belongs to, its day counter and interpolator, and which quotes pin its
pillars. `to_dict()` exposes the full configuration for inspection.

In [8]:
curves = [
    c for c in qs.CurveConfiguration.from_json(str(DATA / "curve_specs.json"))
    if "ESTR" not in repr(c)   # the sample quote file has no ESTR quotes
]
for c in curves:
    cfg = c.to_dict()
    print(f"{cfg['market_index']!s:<22} {cfg['day_counter']:<10} "
          f"{cfg['interpolator']:<18} {len(cfg['quotes'])} quotes")

SOFR                   Actual360  LogLinear          13 quotes
{'Collateral': ('CLP', 'USD')} Actual360  LogLinear          10 quotes
ICP                    Actual360  LogLinear          10 quotes


In [9]:
# FX spot rates and the base discounting configuration of the context.
fx = qs.FxStore.from_dict([{"base": "CLP", "quote": "USD", "rate": 1.0 / 900.0}])
discounting = qs.DiscountingConfig(currency=qs.Currency.USD, index=qs.MarketIndex.SOFR)
discounting

DiscountingConfig(currency=USD, index=SOFR)

## 4. The pricing context

`qs.PricingContext` bundles all market data. **Initializing** it bootstraps
every configured curve, builds volatility surfaces/cubes and runs any
configured simulations.

There are two ways to drive its lifecycle:

- `ctx.initialize()` — bootstrap once, then explore the constructed objects
  freely across notebook cells (used in section 5);
- `with ctx: ...` — additionally starts the automatic-differentiation tape,
  which is required for pricing with sensitivities, and releases everything on
  exit (used in sections 6–7).

In [10]:
ctx = qs.PricingContext(quotes=quotes, curves=curves, fx=fx, discounting=discounting)
ctx.initialize()

ref = ctx.reference_date            # a qs.Date
print("reference date:", ref)
print("original curve configurations:", [str(c.to_dict()['market_index']) for c in ctx.curve_configurations])
print("original quotes:", len(ctx.quotes.identifiers()))

reference date: 2025-11-11
original curve configurations: ['SOFR', "{'Collateral': ('CLP', 'USD')}", 'ICP']
original quotes: 46


## 5. Exploring bootstrapped curves

After initialization the context holds the *constructed* market data. Each
bootstrapped curve is exposed as a `qs.DiscountCurve` view with:

- `nodes()` — the raw date/discount-factor nodes as a DataFrame,
- `discount_factor(date)` and `forward_rate(start, end, compounding, frequency)`,
- `pillars()` — the quote pillars the bootstrap was calibrated to.

(The same pattern applies to `ctx.volatility_surfaces()`, `ctx.volatility_cubes()`
and `ctx.simulations()` when those are configured.)

In [11]:
for curve in ctx.curves():
    print(f"{curve.market_index!s:<22} ref={curve.reference_date}  "
          f"day_counter={curve.day_counter}  nodes={len(curve.nodes())}")

Collateral(CLP/USD)    ref=2025-11-11  day_counter=Actual360  nodes=11
ICP                    ref=2025-11-11  day_counter=Actual360  nodes=11
SOFR                   ref=2025-11-11  day_counter=Actual360  nodes=14


In [12]:
sofr = ctx.curve(qs.MarketIndex.SOFR)
sofr.nodes().head(6)

,date,discount_factor
0,2025-11-11,1.000000
1,2025-11-12,0.999880
2,2026-02-11,0.989043
3,2026-05-11,0.979079
4,2026-11-11,0.959706
5,2027-11-11,0.924388


In [13]:
# Discount factors and forward rates at arbitrary dates.
term = pd.DataFrame(
    {
        "tenor": t,
        "discount_factor": sofr.discount_factor(ref + t),
        "fwd_1y": sofr.forward_rate(ref + t, (ref + t) + "1Y",
                                    qs.Compounding.Simple, qs.Frequency.Annual),
    }
    for t in ["1Y", "2Y", "5Y", "10Y", "20Y"]
)
term

,tenor,discount_factor,fwd_1y
0,1Y,0.959706,0.037683
1,2Y,0.924388,0.037055
2,5Y,0.827320,0.037769
3,10Y,0.683384,0.040522
4,20Y,0.456647,0.033120


In [14]:
# The quote pillars the curve was calibrated to (bootstrap inputs).
sofr.pillars()

,label,value
0,FixedRateDeposit_USD_SOFR_1D,0.04330
1,OIS_USD_SOFR_3M,0.04335
2,OIS_USD_SOFR_6M,0.04250
3,OIS_USD_SOFR_1Y,0.04100
4,OIS_USD_SOFR_2Y,0.03920
5,OIS_USD_SOFR_3Y,0.03840
6,OIS_USD_SOFR_4Y,0.03800
7,OIS_USD_SOFR_5Y,0.03780
8,OIS_USD_SOFR_7Y,0.03770
9,OIS_USD_SOFR_10Y,0.03790


## 6. Pricing trades

Trades are built with typed enums throughout. `ctx.evaluate` takes a list of
`qs.Request` values; sensitivities are computed by automatic differentiation
with respect to the original market quotes, so pricing runs inside a
`with ctx:` block (which starts the AD tape).

In [15]:
swap = qs.Swap(
    identifier="USD_IRS_5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=10_000_000.0,
    fixed_rate=0.0378,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    side=qs.Side.LongReceive,
    fixed_leg_frequency=qs.Frequency.Quarterly,
    floating_leg_frequency=qs.Frequency.Semiannual,
)
print(swap.identifier, "|", swap.currency, swap.market_index, swap.side,
      "|", swap.start_date, "->", swap.maturity_date)

with ctx:
    res = ctx.evaluate(swap, [qs.Request.Value, qs.Request.Cashflows, qs.Request.Sensitivities])
    cashflows = res.cashflows
    sens = res.sensitivities

print(f"NPV = {res.price:,.2f}")
cashflows.head(6)

USD_IRS_5Y | USD SOFR LongReceive | 2025-11-11 -> 2030-11-11
NPV = 8,209.62


,payment_date,type,amount,fixing,accrual_period,currency,caplet_strike,floorlet_strike,leg_index
0,2025-11-11,Disbursement,10000000.0,NaN,0.000000,USD,None,None,0
1,2030-11-11,Redemption,10000000.0,NaN,0.000000,USD,None,None,0
2,2026-02-11,FixedRateCoupon,96600.0,NaN,0.255556,USD,None,None,0
3,2026-05-11,FixedRateCoupon,93450.0,NaN,0.247222,USD,None,None,0
4,2026-08-11,FixedRateCoupon,96600.0,NaN,0.255556,USD,None,None,0
5,2026-11-11,FixedRateCoupon,96600.0,NaN,0.255556,USD,None,None,0


In [16]:
# Quote-level sensitivities (per unit move of each input quote), largest first.
sens.reindex(sens.value.abs().sort_values(ascending=False).index).head(8)

,pillar,value
0,FixedRateDeposit_USD_SOFR_1D,-4.568261e+07
10,OIS_USD_SOFR_5Y,-4.547197e+07
7,OIS_USD_SOFR_3M,-2.414865e+04
11,OIS_USD_SOFR_6M,2.314235e+04
9,OIS_USD_SOFR_4Y,-6.412092e+03
8,OIS_USD_SOFR_3Y,-4.359568e+03
5,OIS_USD_SOFR_2Y,-3.357272e+03
3,OIS_USD_SOFR_1Y,-7.800068e+02


In [17]:
xccy = qs.CrossCurrencySwap(
    identifier="CLPUSD_XCCY_5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    domestic_notional=10_000_000.0,
    foreign_notional=10_000_000.0 * 900.0,
    domestic_currency=qs.Currency.USD,
    foreign_currency=qs.Currency.CLP,
    domestic_market_index=qs.MarketIndex.SOFR,
    foreign_market_index=qs.MarketIndex.ICP,
    foreign_spread=0.002,
    side=qs.Side.LongReceive,
)

with ctx:
    xres = ctx.evaluate(xccy, [qs.Request.Value])
print(f"{xccy.identifier} NPV = {xres.price:,.2f}")

CLPUSD_XCCY_5Y NPV = -80,764,838.31


## 7. XVA — per-client CSA terms and Monte Carlo exposures

The XVA engine simulates portfolio values along Monte Carlo paths and
aggregates CVA/FVA per **netting set**. Each netting set (one client)
carries its own `qs.CsaTerms`:

- `collateral_index` / `collateral_currency` — which curve remunerates posted
  collateral (drives CSA discounting),
- `credit_spread`, `recovery` — the client's credit parameters for CVA,
- `funding_spread` — for FVA,
- optional `credit_index` — a bootstrapped credit curve to use instead of the
  flat spread.

`qs.XvaConfig` holds the simulation setup (paths, seed, model configurations).

In [18]:
xva_config = qs.XvaConfig.from_json(str(DATA / "xva_config.json"))
csa = qs.CsaTerms.from_json(str(DATA / "csa_terms.json"))
print(xva_config)
print(csa)

# CSA terms can also be built inline, per client:
csa_client_b = qs.CsaTerms(
    collateral_index=qs.MarketIndex.SOFR,
    collateral_currency=qs.Currency.USD,
    credit_spread=0.02,
    recovery=0.4,
    funding_spread=0.005,
)

XvaConfig(n_paths=1000, seed=42, models=3)
CsaTerms(collateral_index=SOFR, collateral_currency=USD, credit_spread=0.01, recovery=0.4, funding_spread=0.005)


In [19]:
with ctx:
    result = ctx.run_xva(
        xva_config,
        netting_sets=[
            qs.NettingSet("client_a", [swap, xccy], csa),
            qs.NettingSet("client_b", [swap], csa_client_b),
        ],
    )
    xva_values = result.xva_values
    xva_sens = result.sensitivities
    exposures = result.exposures

xva_values

,netting_set,measure,value
0,client_a,CVA,27140.222643
1,client_a,FVA,715.452999
2,client_b,CVA,8249.178314
3,client_b,FVA,-1049.894898


In [20]:
# Expected positive/negative exposure profiles per netting set.
profile = exposures[0]
print(profile)
profile.to_dataframe().head(8)

ExposureProfile(netting_set='client_a', dates=61)


,date,epe,ene,ee
0,2025-11-11,0.000000,-18687.856374,-18687.856374
1,2025-12-11,114808.909991,-148193.993664,-33385.083673
2,2026-01-11,164556.094401,-220683.485540,-56127.391140
3,2026-02-11,163070.153623,-318208.726220,-155138.572597
4,2026-03-11,181537.146837,-379077.851414,-197540.704577
5,2026-04-11,195648.761766,-435246.423205,-239597.661439
6,2026-05-11,267414.290136,-381731.495076,-114317.204941
7,2026-06-11,302622.727846,-406440.733808,-103818.005963


In [21]:
# XVA sensitivities: market-quote deltas plus per-client credit/funding parameters.
xva_sens.reindex(xva_sens.value.abs().sort_values(ascending=False).index).head(10)

,parameter,value
33,client_a.CVA.credit_spread,2.581969e+06
36,client_b.CVA.credit_spread,3.852707e+05
38,client_b.FVA.funding_spread,-2.099790e+05
35,client_a.FVA.funding_spread,1.430906e+05
34,client_a.CVA.recovery,-2.200888e+03
37,client_b.CVA.recovery,-9.062730e+02
27,OIS_USD_SOFR_3M,0.000000e+00
22,OIS_USD_SOFR_15Y,0.000000e+00
23,OIS_USD_SOFR_1Y,0.000000e+00
24,OIS_USD_SOFR_20Y,0.000000e+00
